In [0]:
%python
from datetime import datetime
import re

# Definir las rutas
#parsing_in = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/stage_pre/20250520"
#last_ingest_time = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/last_ingest_time.txt"

# Definir widgets sin valores por defecto
dbutils.widgets.text("parsing_in_folder", "", "parsing_in")
dbutils.widgets.text("last_ingest_time", "", "Ruta last_ingest_time")

# Obtener los valores de los widgets
parsing_in = dbutils.widgets.get("parsing_in_folder")
last_ingest_time = dbutils.widgets.get("last_ingest_time")

# Listar los archivos en la ruta
try:
    archivos = dbutils.fs.ls(parsing_in)
except Exception as e:
    print(f"Error al acceder a la ruta {parsing_in}: {str(e)}")
    dbutils.notebook.exit("Error: Ruta inválida")

# Función para extraer la fecha del nombre del archivo
def extract_date(filename):
    # Expresión regular para capturar YYYYMMDD_HHMM
    regex = r"TCH_ENCUESTAS_CHECKIN_RESPUESTAS_(\d{8})_(\d{4})\.csv"
    match = re.match(regex, filename)
    if match:
        date_str = f"{match.group(1)}{match.group(2)}"  # Ej. 202505131900
        try:
            # Convertir a datetime (YYYYMMDDHHMM)
            return datetime.strptime(date_str, "%Y%m%d%H%M")
        except ValueError:
            pass
    # Devolver una fecha muy antigua si no coincide
    return datetime(1970, 1, 1)

# Encontrar el archivo con la fecha más reciente
latest_file = None
latest_date = datetime(1970, 1, 1)

for archivo in archivos:
    date = extract_date(archivo.name)
    if date > latest_date:
        latest_date = date
        latest_file = archivo.name

# Formatear la fecha en el formato solicitado (YYYY-MM-DDThh:mm:ss.sssZ)
if latest_file and latest_date != datetime(1970, 1, 1):
    # Agregar milisegundos (000) y formato ISO 8601 con 'Z'
    formatted_date = latest_date.strftime("%Y-%m-%dT%H:%M:%S.000Z")
    print(f"El archivo con la fecha más reciente es: {latest_file}")
    print(f"Fecha formateada: {formatted_date}")
else:
    print("No se encontraron archivos válidos en la ruta especificada.")
    dbutils.notebook.exit("No files found")

# Actualizar el archivo last_ingest_time.txt
try:
    # Crear el contenido con el formato requerido
    new_content = f"timestamp;\n{formatted_date};"
    # Escribir el archivo (sobrescribe el existente)
    dbutils.fs.put(last_ingest_time, new_content, overwrite=True)
    print(f"Archivo {last_ingest_time} actualizado con éxito con la fecha: {formatted_date}")
except Exception as e:
    print(f"Error al actualizar el archivo {last_ingest_time}: {str(e)}")
    dbutils.notebook.exit(f"Error: No se pudo actualizar el archivo")

# Verificar el contenido del archivo actualizado
try:
    content = dbutils.fs.head(last_ingest_time)
    print(f"Contenido actual del archivo {last_ingest_time}:\n{content}")
except Exception as e:
    print(f"Error al leer el archivo {last_ingest_time}: {str(e)}")

In [0]:
%scala
import java.time.LocalDate
import org.apache.spark.sql.{SparkSession, Row}
import org.apache.spark.sql.types._
import scala.util.Try
import scala.collection.JavaConverters._
import org.apache.spark.sql.functions.col
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.DateType
import java.time.format.DateTimeFormatter
import org.apache.spark.sql.{DataFrame, Row}
import org.apache.spark.sql.types.{StructType, StructField, StringType}
import org.apache.spark.sql.Column



In [0]:
%scala

//Calcular las rutas de los últimos 3 días en la data raw.

// Obtener la fecha actual y las fechas de los dos días anteriores
val currentDate = LocalDate.now()
val previousDate = currentDate.minusDays(1)
val twoDaysAgo = currentDate.minusDays(2)

// Extraer año, mes y día como enteros
val yearCurrent = currentDate.getYear
val monthCurrent = f"${currentDate.getMonthValue}%02d"
val dayCurrent = f"${currentDate.getDayOfMonth}%02d"

val yearPrevious = previousDate.getYear
val monthPrevious = f"${previousDate.getMonthValue}%02d"
val dayPrevious = f"${previousDate.getDayOfMonth}%02d"

val yearTwoDaysAgo = twoDaysAgo.getYear
val monthTwoDaysAgo = f"${twoDaysAgo.getMonthValue}%02d"
val dayTwoDaysAgo = f"${twoDaysAgo.getDayOfMonth}%02d"

var basePathLanding = dbutils.widgets.get("basePath_Landing")
//var basePathLanding = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/landing"

// Construcción de rutas con formato year=YYYY/month=M/day=D
val pathCurrentDay = s"$basePathLanding/$yearCurrent/$monthCurrent/$dayCurrent"
val pathPreviousDay = s"$basePathLanding/$yearPrevious/$monthPrevious/$dayPrevious"
val pathTwoDaysAgo = s"$basePathLanding/$yearTwoDaysAgo/$monthTwoDaysAgo/$dayTwoDaysAgo"

// imprimir resultados
println(s"Ruta día actual: $pathCurrentDay")
println(s"Ruta día anterior: $pathPreviousDay")
println(s"Ruta dos días atrás: $pathTwoDaysAgo")


In [0]:
%scala

//Revisar las rutas para rescatar la data y dejarla en dataframes temporales, en el caso que la ruta no exista o no tenga data se crea un dataframe vacío con el esquema de la data.

val spark = SparkSession.builder().appName("DeltaData").getOrCreate()

// Esquema del DataFrame
val schema = StructType(Seq(
  StructField("ID_TURNO", StringType, true),
  StructField("ID_PREGUNTA", StringType, true),
  StructField("VALOR_RESPUESTA", StringType, true),
  StructField("VERBATIM_RESPUESTA", StringType, true),
  StructField("FECHA_HORA_RESPUESTA", StringType, true)
))

// Función para verificar si la ruta tiene archivos
def checkPath(path: String): Boolean = Try(dbutils.fs.ls(path)).getOrElse(Seq()).nonEmpty

// Lista vacía de Row para crear DataFrames vacíos
val emptyList: java.util.List[Row] = Seq.empty[Row].asJava

// Obtener nombres de las columnas del esquema
val selectedColumns = schema.fieldNames.map(col)

// Verificar si las rutas tienen datos y cargar los DataFrames
val dfr1 = if (checkPath(pathCurrentDay)) {
  println(s"La ruta $pathCurrentDay tiene archivos.")
spark.read.format("csv").option("delimiter", ";").option("header", "true").option("inferSchema", "false").schema(schema).load(pathCurrentDay)
} else {
  println(s"La ruta $pathCurrentDay no existe o está vacía. Creando DataFrame vacío...")
  spark.createDataFrame(emptyList, schema)
}

val dfr2 = if (checkPath(pathPreviousDay)) {
  println(s"La ruta $pathPreviousDay tiene archivos.")
spark.read.format("csv").option("delimiter", ";").option("header", "true").option("inferSchema", "false").schema(schema).load(pathPreviousDay)
} else {
  println(s"La ruta $pathPreviousDay no existe o está vacía. Creando DataFrame vacío...")
  spark.createDataFrame(emptyList, schema)
}

val dfr3 = if (checkPath(pathTwoDaysAgo)) {
  println(s"La ruta $pathTwoDaysAgo tiene archivos.")
spark.read.format("csv").option("delimiter", ";").option("header", "true").option("inferSchema", "false").schema(schema).load(pathTwoDaysAgo)
} else {
  println(s"La ruta $pathTwoDaysAgo no existe o está vacía. Creando DataFrame vacío...")
  spark.createDataFrame(emptyList, schema)
}

val mergedDf = dfr1.union(dfr2).union(dfr3).distinct()
//mergedDf.show(10, false)
//mergedDf.printSchema

In [0]:
%scala

var parsing_in = dbutils.widgets.get("parsing_in_folder")
//var parsing_in = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/stage_pre/20250327/"

var parsing_out = dbutils.widgets.get("parsing_out_folder")
//var parsing_out = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/canales/atenciones_suc/tch_encuestas_checkin_respuestas/stage/"

// Función para limpiar columnas String
def cleanStringColumn(colName: String): Column = {
  val trimmedColumn = trim(col(colName))
  when(trimmedColumn === "" || trimmedColumn === "null" || col(colName).isNull, null)
    .otherwise(trimmedColumn)
    .alias(colName)
}

try {

  val TimestampPattern = "yyyy-MM-dd HH:mm:ss"

  // Se leen todos los archivos que existen en la ruta de entrada que han sido extraídos desde el origen.
  val df1 = spark.read.option("header", "true").option("inferSchema", "false").option("delimiter", ";").schema(schema).csv(parsing_in)

  // Se filtran solo los valores que no contienen nulos en las columnas obligatorias y que en la columna FECHA_HORA_RESPUESTA cumplan con el patrón de fecha establecido.

  val df2 = df1.select(df1.columns.map(c => when(trim(col(c)) === "" || col(c) === "null", null).otherwise(col(c)).alias(c)): _*)
    .filter(col("ID_TURNO").isNotNull)
    .filter(col("ID_PREGUNTA").isNotNull)
    .filter(col("FECHA_HORA_RESPUESTA").isNotNull || col("FECHA_HORA_RESPUESTA").rlike(TimestampPattern))
    .dropDuplicates()

  // Realizar el left_anti join entre mi data de entrada y mergedDf para quedarnos solo con los registros que no existen ingestados.
  val df_final = df2.join(mergedDf, usingColumns=Seq("ID_TURNO","ID_PREGUNTA","VALOR_RESPUESTA"), joinType="left_anti")

val rowCount1 = df2.count()
println(s"El número de registros de entrada son: $rowCount1")

val rowCount2 = mergedDf.count()
println(s"El número de registros existentes en raw son: $rowCount2")

val rowCount3 = df_final.count()
println(s"El número de registros nuevos a ingestar son: $rowCount3")

// Construir el nombre del archivo
val currentDate = LocalDate.now()
val formatter = DateTimeFormatter.ofPattern("yyyyMMdd")
val currentDateStr = currentDate.format(formatter)
val filename = s"ENCUESTAS_CHECKIN_RESPUESTAS_$currentDateStr.csv"

  // Guarda el archivo en la carpeta de salida
  df_final.repartition(1).write.option("header", "true").option("delimiter", ";").mode("append").csv(parsing_out)

// Renombrar el archivo guardado con el filename construido
val files = dbutils.fs.ls(parsing_out)
val oldFilePathOption = files.find(file => file.name.startsWith("part-"))

oldFilePathOption match {
  case Some(oldFile) =>
    val oldFilePath = oldFile.path
    val newFilePath = parsing_out + filename
    dbutils.fs.mv(oldFilePath, newFilePath)
    println(s"El archivo ha sido renombrado a: $filename")
  case None =>
    println("No se encontró ningún archivo que cumpla con el criterio.")
}
} catch {
  case e: Exception =>
    println("[ERROR] " + e)
    throw e
}

println("[INFO] PROCESO TERMINADO")

In [0]:
%scala
val files = dbutils.fs.ls(parsing_in)

// Función para eliminar archivos y carpetas temporales que se utilizaron en el proceso.
def deleteFilesAndFolders(path: String): Unit = {
  val filesAndDirs = dbutils.fs.ls(path)

  filesAndDirs.foreach(file => {
    if (file.isFile) {
      dbutils.fs.rm(file.path, true)
    } 
    else if (file.isDir) {
      deleteFilesAndFolders(file.path)
    }
  })
  
  dbutils.fs.rm(path, true)
}

deleteFilesAndFolders(parsing_in)
